In [34]:
import os
PROJECT_DIR = r"E:\Semester 4 (Intelligent System)\Machine Learning\Project"
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Isi folder project:', os.listdir('.'))
print('Isi folder data/:', os.listdir('data'))

Working directory: E:\Semester 4 (Intelligent System)\Machine Learning\Project
Isi folder project: ['data', 'Main.ipynb']
Isi folder data/: ['climate_data.csv', 'climate_data_processed.csv', 'processed', 'station_detail.csv']


In [35]:
import pandas as pd
import numpy as np
import os
import json
import math
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [36]:
INPUT_CSV    = os.path.join('data', 'climate_data.csv')
STATION_CSV  = os.path.join('data', 'station_detail.csv')
OUTPUT_CSV   = os.path.join('data', 'processed', 'climate_data_processed.csv')
MODEL_FILE   = os.path.join('models', 'rf_drought_model.pkl')
ENCODER_FILE = os.path.join('models', 'label_encoder_ddd_car.pkl')
GEOJSON_OUT  = os.path.join('public', 'data', 'drought_risk.geojson')
EDA_DIR      = os.path.join('eda_output')

FEATURES  = ['Tn', 'Tx', 'Tavg', 'RH_avg', 'precipitation', 'ss', 'ff_x', 'ddd_car_code']
GRID_SIZE = 0.25

for folder in ['data/processed', 'models', 'public/data', 'eda_output']:
    os.makedirs(folder, exist_ok=True)

print('Config selesai!')
print(f'Input CSV   : {INPUT_CSV}')
print(f'Station CSV : {STATION_CSV}')
print(f'Output CSV  : {OUTPUT_CSV}')

Config selesai!
Input CSV   : data\climate_data.csv
Station CSV : data\station_detail.csv
Output CSV  : data\processed\climate_data_processed.csv


Prepocessing

In [37]:
def preprocess():
    df = pd.read_csv(INPUT_CSV, parse_dates=['date'])
    stations = pd.read_csv(STATION_CSV)
    print(f'Raw data shape    : {df.shape}')
    print(f'Stations shape    : {stations.shape}')
    # Merge info stasiun
    df = df.merge(
        stations[['station_id', 'station_name', 'region_name', 'latitude', 'longitude']],
        on='station_id',
        how='left'
    )
    # Rename RR -> precipitation
    df['precipitation'] = df['RR']
    # Hitung SPI per stasiun
    df['spi'] = df.groupby('station_id')['precipitation'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    # Encode ddd_car SEKALI, simpan encoder supaya inference pakai mapping identik
    le = LabelEncoder()
    df['ddd_car_code'] = le.fit_transform(df['ddd_car'].fillna('NaN').astype(str))
    os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)
    joblib.dump(le, ENCODER_FILE)
    print(f'LabelEncoder disimpan ke {ENCODER_FILE}')
    # Handle missing values dengan .loc supaya tidak SettingWithCopyWarning
    numerical_cols = ['Tn', 'Tx', 'Tavg', 'RH_avg', 'RR', 'precipitation', 'ss', 'ff_x', 'ff_avg']
    for col in numerical_cols:
        if col in df.columns:
            df.loc[:, col] = df[col].fillna(df[col].median())
    for col in ['ddd_x', 'ddd_car']:
        if col in df.columns:
            df.loc[:, col] = df[col].fillna(df[col].mode()[0])
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f'\n Preprocessing selesai!')
    print(f'Shape setelah merge : {df.shape}')
    print(f'Missing values sisa : {df.isnull().sum().sum()}')
    print(f'Processed data disimpan ke {OUTPUT_CSV}')
    return df
 

EDA

In [38]:
def eda():
    df = pd.read_csv(OUTPUT_CSV)
    df['date'] = pd.to_datetime(df['date'],format='%d-%m-%Y')
    os.makedirs(EDA_DIR, exist_ok=True)
    # 4.1 Info Dasar
    print(f'Rows    : {df.shape[0]:,}')
    print(f'Columns : {df.shape[1]}')
    print(f'Periode : {df["date"].min()} s/d {df["date"].max()}')
    print(f'Stasiun : {df["station_id"].nunique()} stasiun unik')
    print(f'Region  : {df["region_name"].nunique()} region unik')
    print()
    df.info()

    # 4.2 Statistik Deskriptif
    print(df.describe())

    # 4.3 Missing Values
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print('Tidak ada missing values.' if missing.empty else missing)

    # 4.4 Distribusi Kelas Risiko
    def spi_to_risk_label(spi):
        if spi <= -1.0: return 'high'
        elif spi <= 0:  return 'medium'
        else:           return 'low'
    df['risk_label'] = df['spi'].apply(spi_to_risk_label)

    print('\n---DISTRIBUSI KELAS RISIKO ---')
    print(df['risk_label'].value_counts())
    print(f'\nProporsi (%):')
    print((df['risk_label'].value_counts(normalize=True) * 100).round(2))
    fig, ax = plt.subplots(figsize=(6, 4))
    counts = df['risk_label'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=['#e74c3c', '#f39c12', '#2ecc71'])
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'{val:,}', ha='center', fontsize=10)
    ax.set_title('Distribusi Kelas Risiko Kekeringan', fontsize=13)
    ax.set_xlabel('Risk Level')
    ax.set_ylabel('Jumlah Data')
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, 'distribusi_risiko.png'), dpi=150)
    plt.close()
    print('Plot distribusi risiko disimpan.')

    # 4.5 Distribusi Fitur Numerik
    print('\n--- DISTRIBUSI FITUR NUMERIK ---')
    num_cols = [c for c in ['Tn', 'Tx', 'Tavg', 'RH_avg', 'precipitation', 'ss', 'ff_x', 'spi']
                if c in df.columns]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for i, col in enumerate(num_cols):
        axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
        axes[i].set_title(f'Distribusi {col}', fontsize=11)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frekuensi')
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Distribusi Fitur Numerik', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, 'distribusi_fitur.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Plot distribusi fitur disimpan.')
    # 4.6 Heatmap Korelasi
    print('\n---KORELASI ANTAR FITUR ---')
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
    ax.set_title('Korelasi Antar Fitur Numerik', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, 'korelasi_heatmap.png'), dpi=150)
    plt.close()
    print('Heatmap korelasi disimpan.')

    # 4.7 Boxplot Precipitation per Risk Level
    print('\n---BOXPLOT CURAH HUJAN PER RISIKO ---')
    fig, ax = plt.subplots(figsize=(7, 5))
    df.boxplot(column='precipitation', by='risk_label', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title('Distribusi Curah Hujan per Level Risiko', fontsize=13)
    plt.suptitle('')
    ax.set_xlabel('Risk Level')
    ax.set_ylabel('Precipitation (mm)')
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, 'boxplot_precipitation.png'), dpi=150)
    plt.close()
    print('Boxplot precipitation disimpan.')

    # 4.8 Tren SPI Bulanan
    print('\n--- TREN SPI BULANAN ---')
    df['year_month'] = df['date'].dt.to_period('M')
    spi_trend = df.groupby('year_month')['spi'].mean()

    fig, ax = plt.subplots(figsize=(14, 4))
    spi_trend.plot(ax=ax, color='darkorange', linewidth=1.2)
    ax.axhline(0,  color='gray', linestyle='--', linewidth=0.8, label='Normal')
    ax.axhline(-1, color='red',  linestyle='--', linewidth=0.8, label='Batas High Risk (SPI <= -1)')
    ax.fill_between(range(len(spi_trend)),
                    spi_trend.values, -1,
                    where=(spi_trend.values <= -1),
                    alpha=0.3, color='red', label='Zona Kering')
    ax.set_title('Tren SPI Rata-rata Bulanan', fontsize=13)
    ax.set_xlabel('Bulan')
    ax.set_ylabel('SPI')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EDA_DIR, 'tren_spi_bulanan.png'), dpi=150)
    plt.close()
    print('Plot tren SPI disimpan.')
    print(f'\n EDA selesai! Semua plot tersimpan di folder: {EDA_DIR}/')
    print('  - distribusi_risiko.png')
    print('  - distribusi_fitur.png')
    print('  - korelasi_heatmap.png')
    print('  - boxplot_precipitation.png')
    print('  - tren_spi_bulanan.png')

Train Model

In [39]:
def train():
    df = pd.read_csv(OUTPUT_CSV, parse_dates=['date'])
    df = df.dropna(subset=FEATURES)
    # Mengubah nilai SPI menjadi label risiko
    def spi_to_risk(spi):
        if spi <= 0:
            return 1   # medium
        else:
            return 0   # low
    df['risk'] = df['spi'].apply(spi_to_risk)
    # Feature dan target
    X = df[FEATURES]
    y = df['risk']
    # Menampilkan distribusi kelas
    print(f'Distribusi kelas:')
    print(y.value_counts())
    print(f'\nProporsi kelas (%):')
    print((y.value_counts(normalize=True) * 100).round(2))

    # Split data train dan test
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    print(f'\nTraining : {X_train.shape}')
    print(f'Testing  : {X_test.shape}')

    # Membuat model Random Forest
    rf = RandomForestClassifier(n_estimators=80,max_depth=10,min_samples_leaf=10,max_features='sqrt',random_state=42,n_jobs=-1)
    print('Training Random Forest...')
    rf.fit(X_train, y_train)
    # Evaluasi accuracy
    acc = rf.score(X_test, y_test)
    print(f'\nTest Accuracy: {acc:.4f}')
    # Prediksi data test
    y_pred = rf.predict(X_test)
    # Classification Report
    print('\nClassification Report:')
    print(classification_report(y_test,y_pred,target_names=['low (0)', 'medium (1)']))
    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,xticklabels=['low', 'medium'],yticklabels=['low', 'medium'])
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(
        os.path.join(EDA_DIR, 'confusion_matrix.png'),
        dpi=150
    )
    plt.close()
    print('Confusion matrix disimpan.')
    # Feature Importance
    importances = pd.Series(
        rf.feature_importances_,
        index=FEATURES
    ).sort_values(ascending=False)
    print(f'\nFeature Importances:\n{importances}')
    fig, ax = plt.subplots(figsize=(8, 5))
    importances.plot(kind='bar',color='steelblue',ax=ax,edgecolor='white')
    ax.set_title('Feature Importance - Random Forest', fontsize=13)
    ax.set_ylabel('Importance Score')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(
        os.path.join(EDA_DIR, 'feature_importance.png'),
        dpi=150
    )
    plt.close()
    print('Feature importance disimpan.')
    # Simpan model
    joblib.dump(rf, MODEL_FILE)
    print(f'\n Model disimpan ke {MODEL_FILE}')
    return rf

GENERATE GEOJSON

In [40]:
def generate_geojson():
    print('\n========== GENERATE GEOJSON ==========')

    df = pd.read_csv(OUTPUT_CSV, parse_dates=['date'])

    rf = joblib.load(MODEL_FILE)
    le = joblib.load(ENCODER_FILE)  # pakai encoder SAMA dengan training

    df['ddd_car_code'] = le.transform(df['ddd_car'].fillna('NaN').astype(str))
    df = df.dropna(subset=FEATURES)

    print('Memprediksi risiko kekeringan...')
    df['risk'] = rf.predict(df[FEATURES])

    df = df.sort_values('date').groupby('station_id', as_index=False).last()
    df = df.dropna(subset=['latitude', 'longitude'])
    print(f'Total stasiun unik: {len(df)}')

    def grid_id(lat, lon, size):
        return (math.floor(lat / size) * size, math.floor(lon / size) * size)

    df[['grid_lat', 'grid_lon']] = df.apply(
        lambda r: pd.Series(grid_id(r['latitude'], r['longitude'], GRID_SIZE)),
        axis=1
    )

    grid_df = df.groupby(['grid_lat', 'grid_lon']).agg(
        risk=('risk', 'mean'),
        region_name=('region_name', lambda x: x.mode()[0])
    ).reset_index()

    grid_df['risk'] = grid_df['risk'].round().astype(int)

    def risk_label(r):
        if r == 2:   return 'high'
        elif r == 1: return 'medium'
        else:        return 'low'

    grid_df['risk'] = grid_df['risk'].apply(risk_label)

    features = []
    for _, row in grid_df.iterrows():
        lat, lon, size = row['grid_lat'], row['grid_lon'], GRID_SIZE
        polygon = [
            [lon,        lat       ],
            [lon + size, lat       ],
            [lon + size, lat + size],
            [lon,        lat + size],
            [lon,        lat       ]
        ]
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Polygon', 'coordinates': [polygon]},
            'properties': {
                'region_name': row['region_name'],
                'risk': row['risk']
            }
        })

    geojson = {'type': 'FeatureCollection', 'features': features}

    os.makedirs(os.path.dirname(GEOJSON_OUT), exist_ok=True)
    with open(GEOJSON_OUT, 'w') as f:
        json.dump(geojson, f)

    print(f'\nGeoJSON disimpan ke {GEOJSON_OUT}')
    print(f'Total grid cells: {len(features)}')
    print(f'\nDistribusi risiko:')
    print(grid_df['risk'].value_counts())


Main

In [41]:
preprocess()
eda()
train()
generate_geojson()
print('\n========== PIPELINE SELESAI ==========')

Raw data shape    : (589265, 12)
Stations shape    : (192, 7)
LabelEncoder disimpan ke models\label_encoder_ddd_car.pkl

 Preprocessing selesai!
Shape setelah merge : (589265, 19)
Missing values sisa : 125384
Processed data disimpan ke data\processed\climate_data_processed.csv
Rows    : 589,265
Columns : 19
Periode : 2010-01-01 00:00:00 s/d 2020-12-31 00:00:00
Stasiun : 173 stasiun unik
Region  : 147 region unik

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589265 entries, 0 to 589264
Data columns (total 19 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   date           589265 non-null  datetime64[ns]
 1   Tn             589265 non-null  float64       
 2   Tx             589265 non-null  float64       
 3   Tavg           589265 non-null  float64       
 4   RH_avg         589265 non-null  float64       
 5   RR             589265 non-null  float64       
 6   ss             589265 non-null  float64       
 7  